In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, f_oneway, kruskal
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/raw/clima_fortaleza_2016_2026.csv', parse_dates=['data'])

# 1. ESTATÍSTICAS DESCRITIVAS GERAIS

In [3]:
print("="*60)
print("ESTATÍSTICAS DESCRITIVAS - SÉRIE HISTÓRICA 2016-2026")
print("="*60)

estatisticas = df[['precipitacao_mm', 'temperatura_c', 'umidade_percent', 'vento_kmh']].describe()
print(estatisticas)

ESTATÍSTICAS DESCRITIVAS - SÉRIE HISTÓRICA 2016-2026
       precipitacao_mm  temperatura_c  umidade_percent    vento_kmh
count      3834.000000    3834.000000      3834.000000  3834.000000
mean         95.543845      27.752973        73.490532    17.060589
std         111.928695       1.074345         6.212912     5.312801
min           0.000000      24.600000        51.500000     8.500000
25%          23.025000      27.000000        69.300000    13.225000
50%          55.800000      27.700000        73.400000    16.050000
75%         128.000000      28.400000        77.700000    19.700000
max        1007.700000      31.600000        94.400000    54.800000


# 2. ANÁLISE POR ANO

In [4]:
print("\n" + "="*60)
print("ANÁLISE POR ANO")
print("="*60)

analise_anual = df.groupby('ano').agg({
    'precipitacao_mm': ['sum', 'mean', 'max'],
    'temperatura_c': ['mean', 'min', 'max'],
    'umidade_percent': 'mean'
}).round(1)

# Renomear colunas
analise_anual.columns = ['precip_total_mm', 'precip_media_diaria', 'precip_maxima', 
                         'temp_media', 'temp_min', 'temp_max', 'umidade_media']
print(analise_anual)


ANÁLISE POR ANO
      precip_total_mm  precip_media_diaria  precip_maxima  temp_media  \
ano                                                                     
2016          25152.8                 68.7          750.4        28.7   
2017          32753.1                 89.7          870.8        27.5   
2018          29693.6                 81.4          638.4        28.0   
2019          33344.5                 91.4          684.9        27.6   
2020          45173.2                123.4         1007.7        27.1   
2021          46098.4                126.3          790.4        26.9   
2022          41067.5                112.5          875.1        27.4   
2023          24806.9                 68.0          670.4        28.4   
2024          24012.1                 65.6          493.9        28.6   
2025          32745.8                 89.7          671.4        27.7   
2026          31467.2                173.9          726.0        27.0   

      temp_min  temp_max  umidade

# 3. IDENTIFICAR ANOS EXTREMOS

In [5]:
print("\n" + "="*60)
print("ANOS EXTREMOS")
print("="*60)

ano_mais_chuvoso = analise_anual['precip_total_mm'].idxmax()
ano_menos_chuvoso = analise_anual['precip_total_mm'].idxmin()
ano_mais_quente = analise_anual['temp_media'].idxmax()
ano_mais_frio = analise_anual['temp_media'].idxmin()

print(f"🌧️ Ano mais chuvoso: {ano_mais_chuvoso} ({analise_anual.loc[ano_mais_chuvoso, 'precip_total_mm']:.0f} mm)")
print(f"🏜️ Ano menos chuvoso: {ano_menos_chuvoso} ({analise_anual.loc[ano_menos_chuvoso, 'precip_total_mm']:.0f} mm)")
print(f"🔥 Ano mais quente: {ano_mais_quente} ({analise_anual.loc[ano_mais_quente, 'temp_media']:.1f}°C)")
print(f"❄️ Ano mais frio: {ano_mais_frio} ({analise_anual.loc[ano_mais_frio, 'temp_media']:.1f}°C)")



ANOS EXTREMOS
🌧️ Ano mais chuvoso: 2021 (46098 mm)
🏜️ Ano menos chuvoso: 2024 (24012 mm)
🔥 Ano mais quente: 2016 (28.7°C)
❄️ Ano mais frio: 2021 (26.9°C)


# 4. ANÁLISE POR FENÔMENO CLIMÁTICO

In [6]:
print("\n" + "="*60)
print("IMPACTO DOS FENÔMENOS CLIMÁTICOS")
print("="*60)

impacto_fenomenos = df.groupby('fenomeno_climatico').agg({
    'precipitacao_mm': ['mean', 'std'],
    'temperatura_c': ['mean', 'std'],
    'umidade_percent': 'mean'
}).round(1)
print(impacto_fenomenos)


IMPACTO DOS FENÔMENOS CLIMÁTICOS
                   precipitacao_mm        temperatura_c      umidade_percent
                              mean    std          mean  std            mean
fenomeno_climatico                                                          
El Niño Forte                 67.2   85.8          28.6  1.0            68.1
El Niño Fraco                 81.4   92.1          28.0  0.9            72.1
El Niño Moderado              68.0   76.9          28.4  1.0            70.2
La Niña Forte                126.3  137.7          26.9  0.8            80.0
La Niña Fraca                132.8  131.7          27.3  0.8            75.9
La Niña Moderada             123.4  136.9          27.1  0.8            77.9
Neutro                        90.3  104.0          27.6  0.9            73.8


# 5. TESTES ESTATÍSTICOS

In [7]:
print("\n" + "="*60)
print("TESTES DE HIPÓTESE")
print("="*60)

# ANOVA: Diferença significativa entre fenômenos?
grupos_chuva = [group['precipitacao_mm'].values for name, group in df.groupby('fenomeno_climatico')]
f_stat, p_value = f_oneway(*grupos_chuva)
print(f"ANOVA (precipitação vs fenômeno): F-statistic = {f_stat:.2f}, p-value = {p_value:.4f}")
if p_value < 0.05:
    print("✅ Há diferença SIGNIFICATIVA na precipitação entre diferentes fenômenos climáticos")
else:
    print("❌ Não há diferença significativa")

# Correlação entre variáveis
correlacao_temp_precip = pearsonr(df['temperatura_c'], df['precipitacao_mm'])
print(f"\nCorrelação Temperatura x Precipitação: r = {correlacao_temp_precip[0]:.3f} (p={correlacao_temp_precip[1]:.4f})")


TESTES DE HIPÓTESE
ANOVA (precipitação vs fenômeno): F-statistic = 32.98, p-value = 0.0000
✅ Há diferença SIGNIFICATIVA na precipitação entre diferentes fenômenos climáticos

Correlação Temperatura x Precipitação: r = -0.227 (p=0.0000)


# 6. ANÁLISE SAZONAL

In [8]:
print("\n" + "="*60)
print("PADRÕES SAZONAIS")
print("="*60)

analise_mensal = df.groupby('mes').agg({
    'precipitacao_mm': 'mean',
    'temperatura_c': 'mean',
    'umidade_percent': 'mean'
}).round(1)

meses_nomes = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
analise_mensal.index = meses_nomes
print(analise_mensal)

mes_mais_chuvoso = analise_mensal['precipitacao_mm'].idxmax()
mes_mais_seco = analise_mensal['precipitacao_mm'].idxmin()
mes_mais_quente = analise_mensal['temperatura_c'].idxmax()
mes_mais_frio = analise_mensal['temperatura_c'].idxmin()

print(f"\n📅 Mês mais chuvoso: {mes_mais_chuvoso} ({analise_mensal.loc[mes_mais_chuvoso, 'precipitacao_mm']:.1f} mm)")
print(f"📅 Mês mais seco: {mes_mais_seco} ({analise_mensal.loc[mes_mais_seco, 'precipitacao_mm']:.1f} mm)")
print(f"📅 Mês mais quente: {mes_mais_quente} ({analise_mensal.loc[mes_mais_quente, 'temperatura_c']:.1f}°C)")
print(f"📅 Mês mais frio: {mes_mais_frio} ({analise_mensal.loc[mes_mais_frio, 'temperatura_c']:.1f}°C)")



PADRÕES SAZONAIS
     precipitacao_mm  temperatura_c  umidade_percent
Jan             71.2           26.8             73.5
Fev            173.6           27.2             73.6
Mar            215.6           27.4             74.1
Abr            203.8           27.6             73.3
Mai            141.1           27.9             73.9
Jun            101.7           28.1             73.4
Jul             51.6           28.4             73.4
Ago             24.8           28.5             73.4
Set             15.6           28.3             73.4
Out             25.2           28.0             73.7
Nov             34.7           27.7             73.0
Dez             61.3           27.3             73.1

📅 Mês mais chuvoso: Mar (215.6 mm)
📅 Mês mais seco: Set (15.6 mm)
📅 Mês mais quente: Ago (28.5°C)
📅 Mês mais frio: Jan (26.8°C)


# 7. TENDÊNCIAS TEMPORAIS (Regressão Linear)

In [9]:
from scipy.stats import linregress

# Tendência de temperatura
anos = df.groupby('ano')['temperatura_c'].mean().index
temps_anuais = df.groupby('ano')['temperatura_c'].mean().values
slope_temp, intercept, r_value, p_value_temp, std_err = linregress(anos, temps_anuais)

print("\n" + "="*60)
print("TENDÊNCIAS TEMPORAIS")
print("="*60)
print(f"📈 Tendência de temperatura: {slope_temp:.4f}°C/ano")
if slope_temp > 0:
    print(f"   ⚠️ Aquecimento de {slope_temp*10:.2f}°C por década")
    print(f"   R² = {r_value**2:.3f}, p-value = {p_value_temp:.4f}")

# Tendência de precipitação
precip_anuais = df.groupby('ano')['precipitacao_mm'].sum().values
slope_precip, intercept, r_value, p_value_precip, std_err = linregress(anos, precip_anuais)

print(f"📉 Tendência de precipitação: {slope_precip:.1f} mm/ano")
if p_value_precip < 0.05:
    print(f"   Mudança significativa de {slope_precip*10:.0f} mm por década")



TENDÊNCIAS TEMPORAIS
📈 Tendência de temperatura: -0.0363°C/ano
📉 Tendência de precipitação: -60.8 mm/ano


# 8. ANÁLISE DE EXTREMOS CLIMÁTICOS

In [13]:
print("\n" + "="*60)
print("EVENTOS EXTREMOS")
print("="*60)

# Dias com chuva intensa (>50mm)
dias_chuva_intensa = df[df['precipitacao_mm'] > 50]
print(f"🌊 Dias com chuva intensa (>50mm): {len(dias_chuva_intensa)}")
if len(dias_chuva_intensa) > 0:
    dia_mais_chuvoso = dias_chuva_intensa.loc[dias_chuva_intensa['precipitacao_mm'].idxmax()]
    print(f"   Pior dia: {dia_mais_chuvoso['data'].date()} ({dia_mais_chuvoso['precipitacao_mm']:.1f} mm)")
else:
    print("   Nenhum dia com chuva intensa registrado")

# Dias com calor extremo (>32°C)
dias_calor = df[df['temperatura_c'] > 32]
print(f"🔥 Dias com calor extremo (>32°C): {len(dias_calor)}")
if len(dias_calor) > 0:
    dia_mais_quente = dias_calor.loc[dias_calor['temperatura_c'].idxmax()]
    print(f"   Dia mais quente: {dia_mais_quente['data'].date()} ({dia_mais_quente['temperatura_c']:.1f}°C)")
else:
    print("   Nenhum dia com calor extremo registrado")

# Dias com frio atípico (<24°C)
dias_frio = df[df['temperatura_c'] < 24]
print(f"❄️ Dias com temperatura abaixo de 24°C: {len(dias_frio)}")
if len(dias_frio) > 0:
    dia_mais_frio = dias_frio.loc[dias_frio['temperatura_c'].idxmin()]
    print(f"   Dia mais frio: {dia_mais_frio['data'].date()} ({dia_mais_frio['temperatura_c']:.1f}°C)")



EVENTOS EXTREMOS
🌊 Dias com chuva intensa (>50mm): 2051
   Pior dia: 2020-03-01 (1007.7 mm)
🔥 Dias com calor extremo (>32°C): 0
   Nenhum dia com calor extremo registrado
❄️ Dias com temperatura abaixo de 24°C: 0


# 9. COMPARAÇÃO ENTRE PERÍODOS (2016-2020 vs 2021-2026)

In [14]:
print("\n" + "="*60)
print("COMPARAÇÃO ENTRE PERÍODOS")
print("="*60)

# Definindo os períodos corretamente
df_periodo1 = df[df['ano'] <= 2020].copy()  # 2016-2020
df_periodo2 = df[df['ano'] >= 2021].copy()  # 2021-2026

print("Médias 2016-2020 vs 2021-2026:")
print(f"🌡️ Temperatura: {df_periodo1['temperatura_c'].mean():.1f}°C → {df_periodo2['temperatura_c'].mean():.1f}°C "
      f"({df_periodo2['temperatura_c'].mean() - df_periodo1['temperatura_c'].mean():+.2f}°C)")

# Precipitação anual média (total anual / número de anos)
precip_anual_periodo1 = df_periodo1.groupby('ano')['precipitacao_mm'].sum().mean()
precip_anual_periodo2 = df_periodo2.groupby('ano')['precipitacao_mm'].sum().mean()
print(f"💧 Precipitação anual média: {precip_anual_periodo1:.0f} mm → {precip_anual_periodo2:.0f} mm "
      f"({precip_anual_periodo2 - precip_anual_periodo1:+.0f} mm)")

print(f"💨 Umidade média: {df_periodo1['umidade_percent'].mean():.0f}% → {df_periodo2['umidade_percent'].mean():.0f}% "
      f"({df_periodo2['umidade_percent'].mean() - df_periodo1['umidade_percent'].mean():+.1f}%)")

# Teste t para verificar diferença significativa
from scipy.stats import ttest_ind

t_stat, p_value_temp_periodos = ttest_ind(df_periodo1['temperatura_c'], df_periodo2['temperatura_c'])
print(f"\n📊 Diferença de temperatura entre períodos: p-value = {p_value_temp_periodos:.4f}")
if p_value_temp_periodos < 0.05:
    print("   ✅ Diferença ESTATISTICAMENTE SIGNIFICATIVA")
else:
    print("   ❌ Diferença NÃO é estatisticamente significativa")

# Teste para precipitação
t_stat_precip, p_value_precip_periodos = ttest_ind(df_periodo1['precipitacao_mm'], df_periodo2['precipitacao_mm'])
print(f"📊 Diferença de precipitação entre períodos: p-value = {p_value_precip_periodos:.4f}")
if p_value_precip_periodos < 0.05:
    print("   ✅ Diferença ESTATISTICAMENTE SIGNIFICATIVA")
else:
    print("   ❌ Diferença NÃO é estatisticamente significativa")




COMPARAÇÃO ENTRE PERÍODOS
Médias 2016-2020 vs 2021-2026:
🌡️ Temperatura: 27.8°C → 27.7°C (-0.05°C)
💧 Precipitação anual média: 33223 mm → 33366 mm (+143 mm)
💨 Umidade média: 73% → 74% (+0.6%)

📊 Diferença de temperatura entre períodos: p-value = 0.1588
   ❌ Diferença NÃO é estatisticamente significativa
📊 Diferença de precipitação entre períodos: p-value = 0.0147
   ✅ Diferença ESTATISTICAMENTE SIGNIFICATIVA


# 10. ANÁLISE DE CORRELAÇÃO ENTRE VARIÁVEIS

In [15]:
print("\n" + "="*60)
print("CORRELAÇÕES IMPORTANTES")
print("="*60)

# Correlação Temperatura x Umidade
corr_temp_umid = pearsonr(df['temperatura_c'], df['umidade_percent'])
print(f"📉 Correlação Temperatura x Umidade: r = {corr_temp_umid[0]:.3f} (p={corr_temp_umid[1]:.4f})")
if corr_temp_umid[0] < -0.5:
    print("   → Forte relação inversa: quanto mais quente, mais seco")

# Correlação Precipitação x Umidade
corr_precip_umid = pearsonr(df['precipitacao_mm'], df['umidade_percent'])
print(f"💧 Correlação Precipitação x Umidade: r = {corr_precip_umid[0]:.3f} (p={corr_precip_umid[1]:.4f})")

# Correlação Vento x Temperatura
corr_vento_temp = pearsonr(df['vento_kmh'], df['temperatura_c'])
print(f"🌬️ Correlação Vento x Temperatura: r = {corr_vento_temp[0]:.3f} (p={corr_vento_temp[1]:.4f})")



CORRELAÇÕES IMPORTANTES
📉 Correlação Temperatura x Umidade: r = -0.330 (p=0.0000)
💧 Correlação Precipitação x Umidade: r = 0.133 (p=0.0000)
🌬️ Correlação Vento x Temperatura: r = -0.014 (p=0.3743)


# 11. RESULTADOS DETALHADOS POR FENÔMENO

In [16]:
print("\n" + "="*60)
print("RANKING DE FENÔMENOS")
print("="*60)

# Ranking de precipitação
ranking_precip = df.groupby('fenomeno_climatico')['precipitacao_mm'].mean().sort_values(ascending=False)
print("\n🌧️ MAIS CHUVOSO → MENOS CHUVOSO:")
for i, (fen, precip) in enumerate(ranking_precip.items(), 1):
    print(f"   {i}. {fen}: {precip:.1f} mm/dia")

# Ranking de temperatura
ranking_temp = df.groupby('fenomeno_climatico')['temperatura_c'].mean().sort_values(ascending=False)
print("\n🔥 MAIS QUENTE → MAIS FRIO:")
for i, (fen, temp) in enumerate(ranking_temp.items(), 1):
    print(f"   {i}. {fen}: {temp:.1f}°C")


RANKING DE FENÔMENOS

🌧️ MAIS CHUVOSO → MENOS CHUVOSO:
   1. La Niña Fraca: 132.8 mm/dia
   2. La Niña Forte: 126.3 mm/dia
   3. La Niña Moderada: 123.4 mm/dia
   4. Neutro: 90.3 mm/dia
   5. El Niño Fraco: 81.4 mm/dia
   6. El Niño Moderado: 68.0 mm/dia
   7. El Niño Forte: 67.2 mm/dia

🔥 MAIS QUENTE → MAIS FRIO:
   1. El Niño Forte: 28.6°C
   2. El Niño Moderado: 28.4°C
   3. El Niño Fraco: 28.0°C
   4. Neutro: 27.6°C
   5. La Niña Fraca: 27.3°C
   6. La Niña Moderada: 27.1°C
   7. La Niña Forte: 26.9°C


# 12. EXPORTAR RESULTADOS PARA CSV

In [20]:
print("\n" + "="*60)
print("EXPORTANDO RESULTADOS")
print("="*60)

# Salvar análise anual
analise_anual.to_csv('analise_anual.csv')
print("✅ Análise anual salva em 'analise_anual.csv'")

# Salvar análise mensal
analise_mensal.to_csv('analise_mensal.csv')
print("✅ Análise mensal salva em 'analise_mensal.csv'")

# Salvar impacto dos fenômenos
impacto_fenomenos.to_csv('impacto_fenomenos.csv')
print("✅ Impacto dos fenômenos salvo em 'impacto_fenomenos.csv'")

print("\n" + "="*60)
print("✅ ANÁLISE ESTATÍSTICA CONCLUÍDA COM SUCESSO!")
print("="*60)


EXPORTANDO RESULTADOS
✅ Análise anual salva em 'analise_anual.csv'
✅ Análise mensal salva em 'analise_mensal.csv'
✅ Impacto dos fenômenos salvo em 'impacto_fenomenos.csv'

✅ ANÁLISE ESTATÍSTICA CONCLUÍDA COM SUCESSO!
